# 01. 기사 수치 검산과 비교 기준

목표: 기사에 등장하는 정규화 score, 상대 개선율과 비용 배수를 직접 계산합니다. 모든 값은 저자 보고값이며 독립 재현 결과가 아닙니다.

In [ ]:
article = {
    "base_score": 0.642,
    "best_frontier_score": 0.769,
    "specialist_score": 0.873,
    "specialist_cost_per_1k": 0.50,
    "cheapest_frontier_cost_per_1k": 19.0,
    "strong_frontier_cost_per_1k": 34.0,
    "expensive_frontier_cost_per_1k": 172.0,
}

absolute_gain_pp = (article["specialist_score"] - article["best_frontier_score"]) * 100
relative_gain = article["specialist_score"] / article["best_frontier_score"] - 1
gain_over_base_pp = (article["specialist_score"] - article["base_score"]) * 100

print(f"frontier 대비 절대 개선: {absolute_gain_pp:.1f} percentage points")
print(f"frontier 대비 상대 개선: {relative_gain:.1%}")
print(f"base 대비 절대 개선: {gain_over_base_pp:.1f} percentage points")

assert round(absolute_gain_pp, 1) == 10.4
assert round(relative_gain * 100, 1) == 13.5
assert round(gain_over_base_pp, 1) == 23.1

In [ ]:
def cost_ratio(reference_cost_per_1k: float, specialist_cost_per_1k: float) -> float:
    if reference_cost_per_1k <= 0 or specialist_cost_per_1k <= 0:
        raise ValueError("비용은 0보다 커야 합니다.")
    return reference_cost_per_1k / specialist_cost_per_1k

for label in ("cheapest_frontier", "strong_frontier", "expensive_frontier"):
    ratio = cost_ratio(
        article[f"{label}_cost_per_1k"],
        article["specialist_cost_per_1k"],
    )
    print(f"{label:20s}: {ratio:6.1f}x")

assert cost_ratio(19, 0.5) == 38
assert cost_ratio(34, 0.5) == 68
assert cost_ratio(172, 0.5) == 344

In [ ]:
def annual_inference_cost(decisions_per_day: int, cost_per_1k: float) -> float:
    return decisions_per_day * 365 / 1_000 * cost_per_1k

daily_volume = 40_000_000
frontier_annual = annual_inference_cost(daily_volume, 34.0)
specialist_annual = annual_inference_cost(daily_volume, 0.5)

print(f"frontier scenario : ${frontier_annual / 1_000_000:,.1f}M/year")
print(f"specialist scenario: ${specialist_annual / 1_000_000:,.1f}M/year")
print(f"difference         : ${(frontier_annual - specialist_annual) / 1_000_000:,.1f}M/year")

assert round(frontier_annual / 1_000_000, 1) == 496.4
assert round(specialist_annual / 1_000_000, 1) == 7.3

## 해석 주의

- `87.3%`는 단순 accuracy가 아니라 최대 가능 reward에 대한 정규화 score입니다.
- 40x, 68x, 340x는 서로 다른 frontier 가격을 분모로 사용합니다.
- 연환산은 매일 같은 volume과 단가가 유지된다는 scenario이며 labeling, engineering, monitoring과 GPU redundancy를 포함한 TCO가 아닙니다.